In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import format_instructions
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import os

API key validation

In [2]:
from google.genai.errors import APIError
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
try:
    response = llm_gemini.invoke("Hello")
    print(response.content)
except Exception as e:
    print("Raw Error:", e)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPka9RFXFG/rKWZb5FcO3yNknyDEbaDWahBSw3uThTuf0FRkhM0bwNZLGzppzXgPTGBpV8vgqEg2RuDTu+CHzO6R3TIh8o1H6LMka7KGJbpuF5CfPPYcGh'}}]


Envirnoment Variable Checking

In [3]:
if os.environ.get("GOOGLE_API_KEY"):
    print("api key is found that is gemini")
else:
    raise ValueError("GOOGLE_API_KEY environment variable not set")

api key is found that is gemini


Single Chain

In [ ]:
from langchain_core.output_parsers import format_instructions
# task 1
prompt=ChatPromptTemplate.from_messages([
    ("system","you are a teacher"),
    ("user"," Teach me about {topic}")
])

# task 2
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

# task 3
parser=StrOutputParser()

# Chain Creation
chain=prompt|llm_gemini|parser

# chain invocation
result=chain.invoke({"topic": "python"})
result


'Hello! Welcome to your very first Python lesson. I\'m so excited to be your teacher today. \n\nPython is one of the most popular programming languages in the world. It’s used for everything from building websites and analyzing data to creating artificial intelligence (like ChatGPT!). The best part? It is designed to be easy to read and write, almost like plain English.\n\nThink of learning Python like learning a new spoken language, but instead of talking to people, you are giving instructions to a computer.\n\nReady? Let’s begin with Lesson 1!\n\n---\n\n### Lesson 1: Your First "Conversation" with Python\n\nWhen humans want to say hello, we speak. When we want a computer to "speak," we use a command called `print()`. \n\nDon\'t let the word confuse you—it doesn\'t mean printing on a piece of paper. In programming, `print` means **"show this on the screen."**\n\nIf you want the computer to say "Hello, World!", you write it like this:\n\n```python\nprint("Hello, World!")\n```\n\n**Try 

Chain with Custom Function

In [27]:
# custum function
def custom_function(text:str)->str:
     return f"the answer is {text}"

user_input=input("give me a topic to teach")

#chain creation
custom_chain=prompt|llm_gemini|parser|custom_function

#chain invocation
custom_result=custom_chain.invoke({"topic": user_input})
custom_result

'the answer is Hello! Pull up a chair. I am so glad you asked about Artificial Intelligence—commonly called **AI**. It is one of the most fascinating and important topics of our time, and I promise to make it fun and easy to understand. \n\nThink of this as our first lesson. We’ll keep it simple: **What is it, how does it work, and why does it matter?**\n\n---\n\n### 1. What *is* Artificial Intelligence?\nAt its core, **AI is the simulation of human intelligence by machines, especially computer systems.** \n\nNormally, computers only do what humans explicitly program them to do. If you don\'t write code telling a traditional computer to calculate $2 + 2$, it can\'t do it. \n\n**AI is different.** Instead of giving a computer a strict set of rules, we give it *data* and let it learn. It’s the difference between giving someone a fish versus teaching them how to fish.\n\n### 2. How does it actually work?\nYou’ve probably heard terms like **Machine Learning** and **Deep Learning**. Let’s b

parallel chains

In [ ]:
#chain one
# task 1
prompt_One=ChatPromptTemplate.from_messages([
    ("system","you are a teacher"),
    ("user"," Teach me about {topic_one}")
    # {format_instructions}
])

# task 2
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

# task 3
strparser=StrOutputParser()

# task 4 {custum function}
def custom_function(text:str)->str:
     return f"the answer is \n {text}"

#chain two
#task 3
jsonparser=JsonOutputParser()

#task 1
prompt_two = ChatPromptTemplate.from_messages([
    ("system","you are a accurate ai assistent "),
    ("user"," tell me about {topic_two} {format_instructions}"),
    
]).partial(format_instructions=jsonparser.get_format_instructions())

#task 2
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

#chain creation

chain_one=prompt_One | llm_gemini | strparser | custom_function
chain_two = prompt_two | llm_gemini | jsonparser 
 
#chain invacation

parallel_chain = RunnableParallel(
    result1=chain_one,
    result2=chain_two
)
user_input_one=input("enter a topic to teach")
user_input_two=input("enter a topic to know about it")

parallel_chain.invoke({"topic_one":user_input_one,"topic_two":user_input_two})

{'result1': 'the answer is \n Hello! I love this question. Relationships are a very important part of growing up and understanding human connection. \n\nThink of having a girlfriend not as a status symbol, but as having a **close, special friendship** combined with romantic feelings, mutual respect, and care. \n\nLet’s break it down into four main parts: **What it is, How it starts, How to be a good partner,** and **Knowing when you\'re ready.**\n\n---\n\n### 1. What *is* a Girlfriend?\nA girlfriend is someone you choose to spend romantic time with because you enjoy each other’s company, trust one another, and care deeply for each other\'s happiness. It means going from "just friends" to a deeper level of commitment. \n\n### 2. How Do You Get One? (The Foundation)\nYou don\'t "win" a girlfriend like a prize; rather, you build a connection. It usually happens like this:\n* **Friendship First:** You get to know someone and realize you share common interests, laugh at the same things, and

Chain With Passthrough

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import format_instructions
# task 1
prompt=ChatPromptTemplate.from_messages([
    ("system","you are a teacher"),
    ("user"," Teach me about {topic}")
])

# task 2
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

# task 3
parser=StrOutputParser()

# Chain Creation
chain=prompt|llm_gemini|parser

# passthrouth the runnable Parallel
final_chain = RunnableParallel(
    topic=RunnablePassthrough(),
    explanation=chain
)
user_input=input("tell me a topic to teach")

response=final_chain.invoke({"topic":"user_input"})
response

